# Atelier: fine-tune the brief → Brand DNA model (free Colab T4)

This trains a small open model (Qwen2.5-1.5B-Instruct, Apache-2.0) with LoRA to turn a plain-English brief into the Brand DNA JSON that drives Atelier's algorithmic design engine. The model never draws anything: it only sets the dials.

**Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom (about 30–60 minutes).

In [ ]:
# 1. Get the code (if the repo is private, upload a zip instead and unzip it here)
!git clone https://github.com/hacker-git-code/design-studio.git
%cd design-studio
!pip -q install -e ".[llm]"

## 2. (Optional) Teach it your taste
Upload your reference images to `refs/likes/` (and things you dislike to `refs/dislikes/`). If you have captions (pin titles or descriptions), put them in `refs/likes/captions.csv` as `filename,caption`.

In [ ]:
import os
if os.path.isdir('refs/likes'):
    !atelier taste train refs -o taste_models/taste.json
    extra = '--images taste_models/taste.images.jsonl' + (' --captions refs/likes/captions.csv' if os.path.exists('refs/likes/captions.csv') else '')
else:
    extra = ''
print('extra dataset sources:', extra or 'none')

In [ ]:
# 3. Build the dataset (synthetic briefs + your references)
!atelier llm dataset -o llm_data -n 6000 {extra}
# Optional, costs a few dollars of API credit: add real-sounding paraphrases with Claude
# import os; os.environ['ANTHROPIC_API_KEY'] = '...'
# !pip -q install anthropic && atelier llm dataset -o llm_data -n 6000 --paraphrase 1500 {extra}

In [ ]:
# 4. Baseline: how well does the free rule parser do?
!atelier llm eval --data llm_data --backend rules --limit 200

In [ ]:
# 5. Train (LoRA). T4 has no bf16, the script switches to fp16 automatically.
!python -m atelier.llm.train_lora --data llm_data --out checkpoints/atelier-dna --epochs 2 --batch 8 --grad-accum 2

In [ ]:
# 6. Merge the adapter and evaluate: compare with the baseline above
!python -m atelier.llm.merge_export --adapter checkpoints/atelier-dna --out checkpoints/atelier-dna-merged
!atelier llm eval --data llm_data --backend hf:checkpoints/atelier-dna-merged --limit 200

In [ ]:
# 7. Try it
!atelier dna "a moody, rebellious skate label called GRAVEL, black and acid green" --llm hf:checkpoints/atelier-dna-merged

In [ ]:
# 8. Package for Ollama (runs on a laptop CPU) and download
!git clone -q https://github.com/ggml-org/llama.cpp
!pip -q install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py checkpoints/atelier-dna-merged --outfile checkpoints/atelier-dna.gguf --outtype q8_0
!cd checkpoints && zip -q atelier-dna-ollama.zip atelier-dna.gguf Modelfile
from google.colab import files; files.download('checkpoints/atelier-dna-ollama.zip')

On your machine: unzip, then `ollama create atelier-dna -f Modelfile` and `atelier new "your brief" --llm ollama:atelier-dna`.